# Demo — one agent, one deploy target

> **This repository is our ARC-AGI-2 solution.**
> As it stands, running this notebook pushes a private kernel to Kaggle, built
> the way the final submission was: the duck in up to ten takes on the four
> L4s (`TieredTakeDuck.build(environment="kaggle-4xl4")`), from this checkout.
> The push runs the notebook once, for `DEMO_BUDGET_S`; a competition rerun of
> it ignores that and plays the hidden set to the twelve-hour stop line.
> Submitting the kernel to the competition is done on Kaggle, not here.

Two flags: which agent (`duck` / `fliplr`) and where it runs (`slurm` /
`kaggle`). The notebook builds a TAAF benchmark around the agent's own
column, submits it, and stops.

## This is a demo, not "the" way to run a solver

This repo is a **library**. What it offers is the agent columns
(`DuckAgent.build()`, one call per engine/environment coordinate),
`platypus.AgentSolver` / `platypus.LLMAgentSolver`, and TAAF's `Benchmark`
together with its deploy targets. All configuration lives on the agent, so
one fully-configured agent instance describes a whole experiment and there
is nothing left for a launcher to own.

Nothing in the library imports this notebook, and no test depends on it. It
is one wiring of the public pieces, kept short enough to read in a sitting.

## What it does not do

No A/B arms, no multi-pass scoring, no result analysis, no reattachment to a
running job. Those belong wherever the experiment lives, not in the library.

**It also no longer walks every run path.** Inline hosting, the
`simulate_submission` rehearsal and the SLURM bundle are covered by tests
(`packages/taaf/tests/test_deploy_*`, `tests/test_flip_end_to_end.py`); a
real deploy to SLURM or Kaggle is not, and this notebook is how those get
exercised.


In [ ]:
# --- the two flags --------------------------------------------------------
AGENT = "duck"  # "duck" | "fliplr"
TARGET = "kaggle"  # "slurm" | "kaggle"; the final submission's

# Demo budget, both targets. TAAF's own soft-deadline machinery enforces it
# (TufaSlurmTarget.max_runtime_s / KaggleTarget.max_runtime_s), stopping the
# run one buffer short of it. Raise it to run something real.
#
# It runs from worker start, so the engine load comes out of it before the
# first task is played. The duck's two passes of arc2-eval run for hours,
# so at forty minutes it plays their first games; a full run raises this on
# demand.
DEMO_BUDGET_S = 30 * 60  # the final submission's push

# Kaggle machine shape for the LLM agent. The name is the singular
# "NvidiaL4" and it grants the competition's four-L4 machine; anything else
# — including the plausible "NvidiaL4x4" — is not refused at push time, it
# is silently downgraded to a P100, whose compute capability is below what
# PyTorch itself supports, let alone FP8.
KAGGLE_ACCELERATOR = "NvidiaL4"

import datetime
import getpass
import os
import pathlib
import tempfile

import taaf.benchmark
import taaf.deploy_kaggle
import taaf.deploy_slurm
import taaf.standard_benchmarks

import platypus.deployment
from duck import DuckAgent, TieredTakeDuck
from platypus import AgentSolver, LLMAgentSolver
from platypus.reference_agents import FlipAgent

assert AGENT in ("duck", "fliplr"), AGENT
assert TARGET in ("slurm", "kaggle"), TARGET

STAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
# SLURM writes its artifacts from the compute node, so the job dir has to be
# on shared storage. A Kaggle deploy only stages a bundle for upload from
# this machine and nothing reads it afterwards, so it goes to local temp —
# several hundred small files at NFS's ~100 ms per create is minutes of
# deploy time bought for nothing.
RESULTS_ROOT = (
    pathlib.Path("/data/shared/research/arc_2_results") / getpass.getuser()
    if TARGET == "slurm"
    else pathlib.Path(tempfile.gettempdir())
)
JOB_DIR = RESULTS_ROOT / "platypus_demo" / f"{AGENT}_{TARGET}_{STAMP}"

print(f"agent={AGENT} target={TARGET} budget={DEMO_BUDGET_S / 60:.0f} min")
print(f"job_dir={JOB_DIR}")

## What the flags select

**`AGENT`.** `duck` is the solver this repo is for: the model gets the
puzzle as pictures and as numpy arrays in a stateful Python kernel, works
in the kernel, and answers by assigning into `test_out` — read back and
submitted after every call, so a run cut off at a deadline still scores
what the model had reached. It plays `arc2-eval`, the 120 evaluation
tasks, in two passes, so a run long enough to finish (`DEMO_BUDGET_S`) reads
the score with its pass-to-pass spread.

On Kaggle the duck plays every task in up to **ten takes**
(`TieredTakeDuck.build`, `platypus.multi_take`): independent plays of the
task, queued by how far they agree — a game whose takes disagree gets its
next take before one whose takes have settled — and answered by vote. That
is the submission's shape. On SLURM it plays one take: the plain
`DuckAgent.build`.

`fliplr` is free and deterministic: it submits an all-black
grid, then the mirror, twice over, settling on the mirror. Exactly one task
in `arc2-train` is a left-right flip throughout, so what the run should
score is known before it starts — which is what makes it the end-to-end
check for a deploy path. It spends no tokens, so its budget curve has no
x-axis.

**`TARGET`** picks the column: `build()` builds a vLLM engine in the job's
own process tree on one B200; `build(environment="kaggle-4xl4")` is the same
model from a mounted dataset snapshot with vLLM installed `--no-index` from
the wheels kernel and this repo's fork overlaid on it (`vllm_fork.md`,
"Kaggle"), on all four L4s (two pipeline stages of two tensor ranks), with one subprocess per concurrent game
beside the engine — which is why its SLURM job asks for more memory below.
`settings.md` says which layer of the column owns each setting.


In [ ]:
if AGENT == "fliplr":
    agent = FlipAgent()
    solver = AgentSolver(agent=agent)
else:
    # Up to ten takes are the submission's shape; SLURM runs the plain duck.
    agent = (
        DuckAgent.build(environment="slurm-b200")
        if TARGET == "slurm"
        else TieredTakeDuck.build(environment="kaggle-4xl4")
    )
    solver = LLMAgentSolver(agent=agent)

# The duck plays arc2-eval's 120 tasks twice; the flip agent keeps the flip
# demo, whose expected outcome is exact (one task at 100, the rest at 0).
if AGENT == "duck":
    benchmark = taaf.standard_benchmarks.make_benchmark_named("arc2-eval", solver, n_passes=2)
else:
    benchmark = taaf.standard_benchmarks.make_benchmark_flip_demo(solver, n_other=7)
benchmark.label = f"demo_{AGENT}_{TARGET}"
benchmark.job_dir = JOB_DIR
benchmark.submission_path = (
    pathlib.Path("/kaggle/working/submission.json") if TARGET == "kaggle" else JOB_DIR / "submission.json"
)

print(
    f"{benchmark.label}: {len(benchmark.games)} games x {benchmark.n_passes} pass(es), "
    f"max_concurrent={agent.run_config.max_concurrent}, "
    f"process_mode={agent.run_config.process_mode}"
)

## The deploy target

Both take the budget through `max_runtime_s`, and both measure it from
**worker start** — so an engine load comes out of it before a single task is
played.

**SLURM.** One GPU. The walltime is an hour rather than the usual twelve:
`max_runtime_s` is what actually stops the run, and asking the scheduler for
twelve hours to use the demo budget only costs queue priority. The `fliplr`
agent needs no GPU at all, so it asks for none.

`main_project` names the bundled repo the worker `uv sync`s from. It must
equal **this checkout's directory name**, so it is detected rather than
written as a constant that breaks in a worktree.

**Kaggle.** The owner comes from TAAF's own resolver (`.env`, then
`KAGGLE_USERNAME`, then `~/.kaggle/kaggle.json`). The kernel and the source
dataset are named per agent: TAAF's default source dataset is one shared
slug, so two agents deployed from one account would push versions of the
same dataset over each other. The model snapshot and the wheels kernel ride
along on the agent's `run_config`, forwarded by the solver, so there is
nothing to declare for those.

`run_as_submission` stays off, as it did for the final submission: the
push's own run is a rehearsal, and a competition rerun turns it on inside
the notebook. The kernel and its source dataset take names of their own
(`tufa-labs-arc2-solution-<agent>`), so a push from here adds no version to the
kernel that was submitted or to a demo's.

In [ ]:
if TARGET == "slurm":
    # TAAF takes the EARLIER of the budget and the walltime minus a 10-minute
    # teardown buffer, so a budget that reaches the walltime is silently cut
    # short by it instead. The free agent finishes in seconds and asks for a
    # slot on `short`, which the scheduler caps at 30 minutes; the LLM one
    # needs room for a model load before it plays anything.
    walltime, budget_s = ("00:20:00", min(DEMO_BUDGET_S, 300)) if AGENT == "fliplr" else ("01:00:00", DEMO_BUDGET_S)
    walltime_s = int(walltime[:2]) * 3600 + int(walltime[3:5]) * 60
    assert budget_s < walltime_s - 600, "budget >= walltime - 10 min buffer — raise `walltime` too"

    # The submitter's environment is exported to the worker, so a uv cache on
    # login-node-local storage is a path the compute node cannot create.
    shared_cache = pathlib.Path("/data/personal") / os.environ["USER"] / ".cache"
    if not pathlib.Path(os.environ.get("UV_CACHE_DIR", "")).is_relative_to("/data"):
        os.environ["UV_CACHE_DIR"] = str(shared_cache / "uv")
        os.environ["UV_PYTHON_INSTALL_DIR"] = str(shared_cache / "uv-python")

    if AGENT == "fliplr":
        # One spawned child per game, each importing platypus (~130 MB
        # resident). The engine stack is NOT among them — the duck keeps its
        # vLLM import inside `build()` for exactly this reason —
        # but 8 interpreters still need more than a token allocation.
        gpu_kwargs = dict(gpu="CPU", gpu_count=0, total_cpus=4, total_mem_bytes=16 * 1024**3, partition="short")
    else:
        # Weights, the engine, the host KV tier the column claims eagerly at
        # start, and one subprocess per concurrent game, each importing the
        # agent stack. Sized for the heaviest column, Flash-Next on the B200:
        # its 96 GiB tier on top of the PLE table pinned on the host (~48 GiB),
        # and CPUs and memory for the 200 game processes its concurrency runs.
        gpu_kwargs = dict(gpu_count=1, cpus_per_gpu=24, mem_per_gpu_bytes=260 * 1024**3)
    target = taaf.deploy_slurm.TufaSlurmTarget(
        time=walltime,
        max_runtime_s=budget_s,
        main_project=platypus.deployment.detect_main_project(),
        **gpu_kwargs,
    )
else:
    owner = platypus.deployment.kaggle_owner()
    target = taaf.deploy_kaggle.KaggleTarget(
        username=owner,
        kernel_slug=f"tufa-labs-arc2-solution-{AGENT}",
        kernel_title=f"Tufa Labs ARC2 solution {AGENT}",
        dataset_ref=f"{owner}/tufa-labs-arc2-solution-{AGENT}-src",
        cpu_only=(AGENT == "fliplr"),
        # The competition's own hardware. Worth asking for explicitly rather
        # than taking the default shape: an unavailable shape is not refused
        # at push time, it is silently downgraded, and the first sign is the
        # engine refusing the model's quantization on a card too old for it.
        accelerator=None if AGENT == "fliplr" else KAGGLE_ACCELERATOR,
        run_as_submission=False,
        max_runtime_s=DEMO_BUDGET_S,
        # Pinned to the platform's own limit, NOT derived from the demo
        # budget: this notebook is what gets submitted, and a competition
        # rerun ignores DEMO_BUDGET_S (it plays the hidden set to the stop
        # line). Left unset it would resolve to DEMO_BUDGET_S minus the
        # margin, and the real submission would kill itself within the first
        # hour. The margin comes from the module so the two cannot drift.
        submission_stop_runtime_s=12 * 3600.0 - taaf.deploy_kaggle.SUBMISSION_END_MARGIN_S,
        # No start-up deadline, from notebook start or from machine boot (2026-10-30): a
        # submission records the exit one makes as an error and does not score it, and a slow start
        # still shows, as the stop line above landing late, which counts from notebook start.
    )

handle = await benchmark.deploy(target)
print("submitted:", getattr(handle, "job_id", None) or getattr(handle, "kernel_id", handle))
print("job_dir: ", handle.job_dir)

## Fire and forget

Nothing here waits. The job outlives this kernel, and re-running the
notebook always launches a **new** job into a new timestamped directory — it
never reattaches to a running one, so an accidental re-run costs an
allocation but can never disturb a run in flight.

**The results are `diagnostics.html` in the job dir** — TAAF writes it when
the run finishes, with the scores, the per-game breakdown and the answer
sheets. `make viewer` shows a run while it is still going: it serves
`platypus.viewer` over the results tree and renders from the diagnostic
streams on every request, so an in-flight run is readable down to the LLM
turn currently running.

For **SLURM** that is the job dir printed above, on shared storage,
alongside `stdout.log` / `stderr.log` and the per-game streams under
`games/`. For **Kaggle** the run happens there, so there is nothing local to
watch and the durable copy lives on Kaggle; the local job dir only ever held
the uploaded bundle and it is in temp. `handle.wait()` pulls the finished
output down, and can pull it again from the kernel id at any time.

The cell below is optional — it blocks until the job stops.

In [ ]:
import asyncio

# handle.wait() polls and blocks, so run it off the event loop.
finished = await asyncio.get_running_loop().run_in_executor(None, handle.wait)
for run in sorted(finished.game_runs, key=lambda r: -(r.final_score or 0)):
    tokens = taaf.diagnostics.total_tokens(run)
    print(f"{run.game_id}  {run.state:<10} score={run.final_score}  answers={len(run.history)}  tokens={tokens}")